# Cursul 6. Analiza exploratorie
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

**Analiza exploratorie** este examinarea datelor înainte de modelare (Tukey, 1977): dacă sunt ceea ce se presupune că sunt, cum variază fiecare variabilă, ce variază împreună, ce iese din tipar. Analiștii o fac în aceeași ordine ([Wongsuphasawat, Liu și Heer, 2019](https://ar5iv.arxiv.org/html/1911.00568)): întâi profilul datelor, apoi descoperirea.

| Etapa | Conceptele |
|---|---|
| 1. Profilul datelor | unitatea de analiză, lipsurile, extremele |
| 2. O variabilă | media și mediana; distribuția; transformarea logaritmică |
| 3. Variabile categorice | tabelul de contingență |
| 4. Două variabile numerice | Pearson, Spearman, Kendall; asemănarea economică |
| 5. Granița și distanța | factorul de confuzie, stratificarea |
| 6. Pe hartă | partenerii peste așteptare ai unei regiuni |
| 7. Rețeaua | comunitățile Louvain |
| 8. Verificarea | relație reală sau întâmplare (permutarea) |

Fiecare concept este definit pe scurt și aplicat pe datele cursului: perechile de regiuni cu SCI și tabelul regiunilor. Celulele **Exercițiu** cer câte un rând de cod, iar paragrafele **De încercat** propun câte o modificare; manualul lecției descrie aceiași pași.

Un tipar găsit prin explorare este provizoriu: aceleași date nu pot genera o ipoteză și totodată să o confirme (Behrens, 1997: acuzarea, nu condamnarea). Verificarea din secțiunea 8 arată doar dacă un tipar se deosebește de întâmplare; confirmarea cere alte date.

## 0. Pregătirea
Bibliotecile, datele din C5 și distanța dintre centrele regiunilor (contururile GISCO, ca în C3).

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from scipy import stats
from scipy.stats.contingency import association
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
engine = create_engine("sqlite:///data/dsw.db")    # baza din C5

regiuni = pd.read_sql("select * from regiuni", engine)
perechi = pd.read_sql("select * from perechi_regiuni", engine)
tari = pd.read_sql("select * from perechi_tari", engine)

harta = gpd.read_file("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_3035_LEVL_2.geojson")               # contururile regiunilor, ca în C3
harta = harta.rename(columns={"NUTS_ID": "regiune"})
harta = harta[["regiune", "geometry"]].merge(regiuni, on="regiune")

In [ ]:
# centrul fiecărei regiuni; harta GISCO are coordonatele în metri
centre = harta.set_index("regiune").centroid

# distanța dintre centrele celor două regiuni din fiecare pereche, în kilometri
centru_1 = centre[perechi["regiune_1"]].values
centru_2 = centre[perechi["regiune_2"]].values
perechi["distanta_km"] = centru_1.distance(centru_2) / 1000

# intervalele de distanță și apartenența la aceeași țară
perechi["interval_km"] = pd.cut(perechi["distanta_km"], [0, 100, 250, 500, 1000, 2000, 10000])
perechi["aceeasi_tara"] = perechi["tara_1"] == perechi["tara_2"]

## 1. Profilul datelor
### 1.1. Ce este un rând și ce tipuri de variabile are
**Unitatea de analiză** este ce reprezintă un rând. Tipul fiecărei coloane decide ce rezumat și ce grafic au sens: identificator (nu se rezumă), numerică (medie, mediană, histogramă), categorică nominală (frecvențe, bare), categorică ordinală (frecvențe în ordinea categoriilor, mediana), binară (o proporție).

In [ ]:
perechi[["regiune_1", "tara_1", "distanta_km", "interval_km", "aceeasi_tara", "sci"]].head()

In [ ]:
perechi[["regiune_1", "tara_1", "distanta_km", "interval_km", "aceeasi_tara", "sci"]].dtypes

Un rând este o **pereche de regiuni**: 298 de regiuni, fiecare cu toate celelalte, o singură dată. Fiecare regiune apare în 297 de perechi, deci perechile nu sunt observații independente. `regiune_1` e un identificator, `tara_1` nominală, `distanta_km` și `sci` numerice, `interval_km` ordinală, `aceeasi_tara` binară.

In [ ]:
len(perechi), 298 * 297 // 2

In [ ]:
regiuni[regiuni["tara"] == "RO"]["populatie"].sum()        # un total, față de o sursă externă: ~19 milioane

**Exercițiu.** Același control pentru Ungaria: populația totală a regiunilor ei (aproximativ 9,6 milioane).

In [ ]:
# Suma populației regiunilor din Ungaria: ca pentru România, cu tara "HU"

### 1.2. Lipsurile: unde și de ce
O valoare poate lipsi **complet aleator** (lipsa nu depinde de nimic), **aleator condiționat** (depinde de o variabilă cunoscută, de exemplu țara, adică sursa) sau **neîntâmplător** (depinde de valoarea care lipsește). În primul caz, un rezultat calculat fără ele e doar mai puțin precis; în al doilea, e corect pentru grupurile care au date; în al treilea, e deplasat (Rubin, 1976).

In [ ]:
coloane = ["populatie", "pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "densitate"]
regiuni[coloane].isna().sum()                           # câte lipsesc, pe indicatori

Numărul spune puțin; contează unde sunt. PIB-ul lipsește pe țări întregi: lipsa depinde de țară, nu de întâmplare.

In [ ]:
regiuni[regiuni["pib_locuitor"].isna()]["tara"].value_counts()

Pe hartă: câți dintre cei șase indicatori lipsesc în fiecare regiune.

In [ ]:
harta["lipsa"] = harta[coloane].isna().sum(axis=1)       # câți indicatori lipsesc, pe regiune
ax = harta.plot(column="lipsa", cmap="Reds", legend=True, figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Indicatori lipsă, pe regiuni")
ax.set_axis_off()

Consecința: o comparație pe PIB pierde Elveția și Norvegia, două țări bogate. Fiecare rezultat spune pe câte regiuni a fost calculat.

**De încercat.** Pentru `somaj`: se compară numărul regiunilor cu valori lipsă din fiecare țară cu numărul total de regiuni al țării (`regiuni["tara"].value_counts()`). Lipsesc țări întregi sau regiuni izolate?

### 1.3. Extremele: valori reale sau artefacte?
Primele și ultimele valori se citesc una câte una. SCI este o **rată**: prieteniile dintre două regiuni împărțite la numărul de utilizatori. Pe un numitor foarte mic, câteva prietenii în plus sau în minus duc rata la un capăt sau la celălalt, deci unitățile foarte mici ajung la ambele capete ale clasamentului (Wainer, 2007).

In [ ]:
perechi.nlargest(5, "sci")[["regiune_1", "regiune_2", "distanta_km", "sci"]]

In [ ]:
perechi.nsmallest(5, "sci")[["regiune_1", "regiune_2", "distanta_km", "sci"]]

In [ ]:
regiuni[regiuni["regiune"] == "NO0B"]

Svalbard (`NO0B`), cu câteva mii de locuitori, este la ambele capete. Nu e o descoperire, ci efectul numitorului mic: se exclude din model.

**Exercițiu.** Primele cinci perechi după SCI, fără perechile care conțin Svalbard (`NO0B`).

In [ ]:
# 1. Perechile în care nici regiune_1, nici regiune_2 nu este "NO0B" (două condiții, cu &)
# 2. Primele 5 după sci (nlargest), cu coloanele regiune_1, regiune_2, distanta_km, sci

## 2. O variabilă numerică
### 2.1. Media sau mediana?
**Media** folosește mărimea fiecărei valori, deci câteva valori foarte mari o mută mult. **Mediana** (valoarea din mijloc) depinde doar de ordine și aproape nu se mișcă. La fel pentru împrăștiere: **abaterea standard** (sensibilă) și **intervalul intercuartilic, IQR** (lățimea jumătății din mijloc, robust). 

In [ ]:
sci = perechi["sci"]
print("media:            ", round(sci.mean()))
print("mediana:          ", round(sci.median()))
print("abaterea standard:", round(sci.std()))
print("IQR:              ", round(sci.quantile(0.75) - sci.quantile(0.25)))

Media SCI este de opt ori mai mare decât mediana, din cauza câtorva perechi cu valori foarte mari. Fără cele 8 perechi cu cele mai mari valori:

In [ ]:
fara = sci.drop(sci.nlargest(8).index)                 # fără cele 8 cele mai mari valori
print("media:  ", round(sci.mean()), "->", round(fara.mean()))
print("mediana:", round(sci.median()), "->", round(fara.median()))

**Exercițiu.** Media și mediana distanței (`distanta_km`). Raportul dintre ele este mai apropiat de 1 decât la SCI? Ce spune despre forma distribuției?

In [ ]:
# Media (mean) și mediana (median) coloanei distanta_km, rotunjite

### 2.2. Distribuția: aceleași date, mai multe grafice
Histograma depinde de numărul de bare; densitatea, de netezire; **ECDF** (proporția valorilor sub fiecare prag) nu cere nicio alegere; boxplotul rezumă în cuartile și ascunde forma.

In [ ]:
sns.histplot(perechi, x="sci", log_scale=True, bins=40);       # histograma; bins = numărul barelor

In [ ]:
sns.ecdfplot(perechi, x="sci", log_scale=True);                # ECDF: proporția perechilor cu SCI ≤ x

In [ ]:
(perechi["sci"] <= 1000).mean()                                # aceeași idee, ca număr: proporția sub 1.000

Vârsta medie a regiunilor: boxplotul arată doar cuartilele; violinul cu puncte arată și coada spre stânga, regiunile tinere din Turcia.

In [ ]:
sns.boxplot(regiuni, x="varsta_medie");

In [ ]:
sns.violinplot(regiuni, x="varsta_medie", inner="point");      # inner="point": și fiecare regiune

**De încercat.** Histograma SCI cu `bins=8`, `20`, `80`; densitatea cu `bw_adjust=0.2` și `2`. Care ascund forma și care adaugă zgomot? Regiunile din coada stângă a vârstei: `regiuni.nsmallest(10, "varsta_medie")`.

### 2.3. Transformarea logaritmică
SCI merge de la 1 la peste 4 milioane. **Transformarea logaritmică** înlocuiește fiecare valoare cu log10 x (10 devine 1, 1.000 devine 3, 1.000.000 devine 6): ordinea se păstrează, valorile mari se comprimă, iar diferențele devin rapoarte. **Asimetria** (engl. *skewness*) măsoară dezechilibrul distribuției: 0 pentru o distribuție simetrică. Se alege transformarea cea mai slabă care o aduce aproape de 0, în ordinea x, √x, log x, −1/x (scara lui Tukey, engl. *ladder of powers*).

In [ ]:
print("asimetria lui x:    ", round(sci.skew(), 2))
print("asimetria lui √x:   ", round(np.sqrt(sci).skew(), 2))
print("asimetria lui log x:", round(np.log10(sci).skew(), 2))

In [ ]:
sns.histplot(x=np.log10(perechi["sci"]));                      # histograma lui log10(SCI)

Aceeași idee pentru migranți: ca număr, depind de mărimea țărilor; ca **rată** la 1.000 de locuitori, se compară cu SCI, care e și el o măsură relativă.

In [ ]:
# populația fiecărei țări în 2024, din datele Băncii Mondiale (C2)
bm = pd.read_csv("data/raw/bm.csv", keep_default_na=False, na_values=[""])   # NA = Namibia, nu valoare lipsă
pop = bm[(bm["indicator"] == "populatie") & (bm["an"] == 2024)].set_index("tara")["valoare"]

# migranții la 1.000 de locuitori ai celor două țări
cu_migratie = tari.dropna(subset=["migranti"]).copy()
populatia_ambelor = cu_migratie["tara_1"].map(pop) + cu_migratie["tara_2"].map(pop)
cu_migratie["la_mie"] = 1000 * cu_migratie["migranti"] / populatia_ambelor

In [ ]:
cu_migratie["migranti"].corr(cu_migratie["sci"], method="spearman")     # cu numărul de migranți

In [ ]:
cu_migratie["la_mie"].corr(cu_migratie["sci"], method="spearman")       # cu rata la mie

**De încercat.** Aceleași trei asimetrii pentru `perechi["distanta_km"]` și pentru `regiuni["densitate"]`: ce transformare este suficientă? Apoi zborurile: `cu_migratie["pasageri"].corr(cu_migratie["sci"], method="spearman")`. Sunt legate de SCI mai mult sau mai puțin decât migranții?

## 3. Variabile categorice: tabelul de contingență
Două variabile categorice se rezumă într-un **tabel de contingență**: câte cazuri are fiecare combinație. Procentele se calculează pe direcția întrebării: pe rând („dintre perechile aflate la o distanță, câte sunt în aceeași țară?”) sau pe coloană.

In [ ]:
pd.crosstab(perechi["interval_km"], perechi["aceeasi_tara"])                      # numere

In [ ]:
pd.crosstab(perechi["interval_km"], perechi["aceeasi_tara"], normalize="index").round(3)   # % pe rând

Sub 100 km, trei sferturi din perechi sunt în aceeași țară; peste 1.000 km, sub 1%.

Cât de puternică este legătura? Pentru două variabile categorice, **testul χ²** (hi-pătrat) spune dacă legătura se deosebește de întâmplare (valoarea p), iar **V al lui Cramér** spune cât de puternică este: 0 înseamnă nicio legătură, 1 o legătură perfectă. Ca reper (Cohen, 1988): sub 0,1 neglijabilă, 0,1-0,3 slabă, 0,3-0,5 moderată, peste 0,5 puternică.

In [ ]:
tabel = pd.crosstab(perechi["interval_km"], perechi["aceeasi_tara"])
print("p, testul χ²:", stats.chi2_contingency(tabel).pvalue)
print("V Cramér:    ", round(association(tabel, method="cramer"), 2))

V = 0,48: distanța și granița sunt puternic legate. Consecința se vede în secțiunea 5.

**Exercițiu.** Același tabel, cu procentele pe coloană (`normalize="columns"`). Propoziția pentru celula „0-100 km, aceeași țară” se compară cu citirea pe rând.

In [ ]:
# Ca tabelul cu % pe rând, dar cu normalize="columns"

## 4. Două variabile numerice
### 4.1. Pearson, Spearman, Kendall
**Pearson**: cât de aproape de o dreaptă sunt punctele; e tras de valorile extreme. **Spearman**: Pearson pe ranguri, deci relația monotonă. **Kendall**: proporția perechilor de observații în aceeași ordine, minus cele în ordine inversă. Spearman și Kendall nu se schimbă la logaritm, pentru că logaritmul păstrează ordinea; Pearson se schimbă.

In [ ]:
perechi["distanta_km"].corr(perechi["sci"], method="spearman")           # Spearman

In [ ]:
perechi["distanta_km"].corr(perechi["sci"])                              # Pearson, pe valorile brute

In [ ]:
np.log10(perechi["distanta_km"]).corr(np.log10(perechi["sci"]))          # Pearson, pe log

In [ ]:
perechi["distanta_km"].corr(perechi["sci"], method="kendall")            # Kendall

**De încercat.** Cele trei metode pentru `pib_locuitor` și `densitate` în tabelul regiunilor, apoi Pearson pe `np.log10(regiuni["densitate"])`. Care coeficient se schimbă?

### 4.2. Se leagă regiunile asemănătoare economic?
Ipoteza: regiunile cu PIB pe locuitor apropiat au mai multe legături. **Decalajul** unei perechi este diferența absolută dintre PIB-urile celor două regiuni. Se folosesc doar perechile cu PIB la ambele capete (secțiunea 1.2).

In [ ]:
economice = perechi.dropna(subset=["pib_locuitor_1", "pib_locuitor_2"])     # perechile cu PIB la ambele capete
len(economice), len(perechi)

In [ ]:
decalaj = (economice["pib_locuitor_1"] - economice["pib_locuitor_2"]).abs()   # diferența de PIB, fără semn
decalaj.corr(economice["sci"], method="spearman")

Corelația este slab pozitivă (în jur de 0,16), pe 37.950 din 44.253 de perechi: perechile cu decalaj mai mare sunt, în medie, puțin **mai** legate, contrar ipotezei. Bailey și colegii (2020) găsesc același semn; o explicație posibilă este migrația, din regiunile mai sărace spre cele mai bogate. Rezultatul se spune împreună cu numărul de perechi, pentru că Elveția și Norvegia lipsesc.

**De încercat.** Aceeași corelație doar pentru perechile din țări diferite (`economice[~economice["aceeasi_tara"]]`). Se schimbă semnul sau doar mărimea?

## 5. Granița și distanța: contează granița sau doar distanța?
Perechile din aceeași țară au SCI mult mai mare. Dar ele sunt aproape toate apropiate (secțiunea 3), iar apropierea crește singură SCI-ul. Ca să vedem efectul graniței, comparăm doar perechi aflate la aceeași distanță. Distanța este aici un **factor de confuzie** (engl. *confounder*): o a treia variabilă care încurcă comparația.

Pasul 1: comparația directă, pe toate perechile.

In [ ]:
mediane = perechi.groupby("aceeasi_tara")["sci"].median()
mediane[True] / mediane[False]          # de câte ori mai mare în aceeași țară

Pasul 2: doar perechile aflate la 250-400 km. Aici distanța e aproape aceeași pentru ambele grupuri, deci diferența care rămâne nu mai vine din distanță.

In [ ]:
la_fel = perechi[perechi["distanta_km"].between(250, 400)]
mediane = la_fel.groupby("aceeasi_tara")["sci"].median()
mediane[True] / mediane[False]          # de câte ori mai mare, la aceeași distanță

Pasul 3: aceeași comparație pe fiecare interval de distanță (**stratificarea**).

In [ ]:
mediane = perechi.groupby(["interval_km", "aceeasi_tara"], observed=True)["sci"].median().unstack()
mediane[True] / mediane[False]

In [ ]:
sns.boxplot(perechi, x="interval_km", y="sci", hue="aceeasi_tara", log_scale=True);

La aceeași distanță, raportul e mult mai mic decât pe toate perechile, dar rămâne mult peste 1: granița contează și separat de distanță. În C7, distanța și granița intră amândouă în model.

**De încercat.** Aceeași comparație la 80-130 km și la 600-1.000 km.

## 6. Pe hartă: partenerii peste așteptare
SCI scade puternic cu distanța, deci legăturile brute ale unei regiuni arată mai ales vecinii. Ca să se vadă ce iese din tipar, SCI-ul fiecărei perechi se împarte la **mediana perechilor din același interval de distanță**. Raportul 1 înseamnă exact cât sugerează distanța; 20, de 20 de ori mai mult.

In [ ]:
mediana_intervalului = perechi.groupby("interval_km", observed=True)["sci"].transform("median")
perechi["fata_de_distanta"] = perechi["sci"] / mediana_intervalului

Partenerii unei regiuni: perechile în care apare, ca prima sau ca a doua regiune. Se păstrează partenerii din alte țări, cei mai legați peste așteptare.

In [ ]:
REGIUNE = "RO11"                                      # Nord-Vest

ca_1 = perechi[perechi["regiune_1"] == REGIUNE].rename(columns={"regiune_2": "partener", "tara_2": "tara"})
ca_2 = perechi[perechi["regiune_2"] == REGIUNE].rename(columns={"regiune_1": "partener", "tara_1": "tara"})
legaturi = pd.concat([ca_1, ca_2])[["partener", "tara", "distanta_km", "sci", "fata_de_distanta"]]

externi = legaturi[legaturi["tara"] != REGIUNE[:2]].nlargest(10, "fata_de_distanta")
externi

In [ ]:
ax = harta.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))      # fundalul: toate regiunile
for partener in externi["partener"]:                                     # o linie spre fiecare partener
    ax.plot([centre[REGIUNE].x, centre[partener].x], [centre[REGIUNE].y, centre[partener].y], color="#c0392b")
harta[harta["regiune"] == REGIUNE].plot(ax=ax, color="#c0392b")          # regiunea aleasă
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title=f"{REGIUNE}: partenerii peste așteptare")
ax.set_axis_off()

Partenerii peste așteptare ai regiunii Nord-Vest nu sunt vecinii: sunt regiuni din Spania (La Rioja, Castilla-La Mancha, Aragón), Irlanda, Bruxelles și Austria, de 20 până la 60 de ori mai legate decât sugerează distanța. Sunt țările migrației românești. În C7, aceste perechi se urmăresc în reziduurile modelului.

Ghicitoarea din C1: cu ce țări e România cel mai legată? Perechile României, cu SCI-ul, migranții și rata lor.

In [ ]:
cu_ro = cu_migratie[(cu_migratie["tara_1"] == "RO") | (cu_migratie["tara_2"] == "RO")]
cu_ro.sort_values("sci", ascending=False)[["tara_1", "tara_2", "sci", "migranti", "la_mie"]].head(8)

Italia are cei mai mulți migranți români (peste un milion) și cea mai mare rată, dar este abia a cincea după SCI: Austria, Germania, Ungaria și Irlanda sunt mai legate. Ungaria are puțini migranți români, dar o minoritate maghiară numeroasă în România. Migrația explică mult din SCI, dar nu tot.

**De încercat.** Aceeași hartă pentru `"LU00"` (Luxemburg), `"PL91"` (Varșovia) sau regiunea echipei.

## 7. Rețeaua și comunitățile
Tabelul perechilor este o **rețea**: fiecare regiune este un nod, fiecare pereche o legătură, iar SCI-ul este greutatea legăturii. O **comunitate** este un grup de regiuni mai legate între ele decât cu restul. Algoritmul **Louvain** (Blondel și colegii, 2008) le caută folosind doar greutățile: nu știe ce țară are fiecare regiune și nici unde se află.

In [ ]:
G = nx.from_pandas_edgelist(perechi, "regiune_1", "regiune_2", edge_attr="sci")   # nodurile și legăturile
G.number_of_nodes(), G.number_of_edges()

In [ ]:
grupuri = nx.community.louvain_communities(G, weight="sci", seed=1)   # o listă de grupuri de regiuni

comunitate = {}                                   # regiune -> numărul grupului ei
for numar, grup in enumerate(grupuri):
    for regiune in grup:
        comunitate[regiune] = numar

harta["comunitate"] = harta["regiune"].map(comunitate)
len(grupuri)

In [ ]:
ax = harta.plot(column="comunitate", categorical=True, cmap="tab20", figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Comunitățile rețelei SCI")
ax.set_axis_off()

Aceeași rețea, desenată fără hartă. Ca desenul să se poată citi, se păstrează doar cele mai puternice 5 legături ale fiecărei regiuni. Pozițiile le alege `spring_layout`: regiunile legate între ele se atrag, celelalte se resping. Eticheta e țara, culoarea e comunitatea.

In [ ]:
# fiecare pereche de două ori: o dată din partea fiecărei regiuni
ambele = pd.concat([perechi.rename(columns={"regiune_1": "regiune", "regiune_2": "partener"}),
                    perechi.rename(columns={"regiune_2": "regiune", "regiune_1": "partener"})])
puternice = ambele.sort_values("sci", ascending=False).groupby("regiune").head(5)   # primele 5 ale fiecărei regiuni

R = nx.from_pandas_edgelist(puternice, "regiune", "partener")
pozitii = nx.spring_layout(R, seed=1)                       # așezarea nodurilor
plt.figure(figsize=(10, 8))
nx.draw(R, pozitii, node_size=200, node_color=[comunitate[r] for r in R.nodes], cmap="tab20",
        edge_color="#dddddd", labels={r: r[:2] for r in R.nodes}, font_size=6)

16 comunități. Culorile sunt doar etichete: comunitatea 3 nu este „mai mult” decât comunitatea 2. Pe hartă și în desenul rețelei, cele mai multe urmează granițele, deși algoritmul nu le cunoaște. Câteva le trec: România cu Germania, Austria cu fosta Iugoslavie, Polonia cu Irlanda și Lituania, Portugalia cu Luxemburg. Sunt coridoarele din secțiunea 6. Gruparea arată *că* unele regiuni sunt legate peste granițe, nu și *de ce*: migrația, limba și istoria se verifică cu alte date.

**De încercat.** `resolution=0.5`, apoi `resolution=2`, în `louvain_communities`: câte comunități ies și cum se schimbă harta?

## 8. Verificarea: relație reală sau întâmplare?
Orice corelație calculată pe date reale iese diferită de zero, chiar între variabile fără nicio legătură. Două numere răspund la două întrebări diferite:
- **intensitatea**: coeficientul însuși (ρ), cât de puternică este relația;
- **valoarea p**: cât de des ar ieși o corelație cel puțin la fel de mare dacă nu ar exista nicio legătură. Sub 0,05, relația se consideră reală (**semnificativă statistic**).

`stats.spearmanr` le dă pe amândouă; `nan_policy="omit"` lasă deoparte regiunile fără date.

In [ ]:
stats.spearmanr(regiuni["pib_locuitor"], regiuni["studii_superioare"], nan_policy="omit")

In [ ]:
stats.spearmanr(regiuni["varsta_medie"], regiuni["studii_superioare"], nan_policy="omit")

In [ ]:
stats.spearmanr(regiuni["pib_locuitor"], regiuni["varsta_medie"], nan_policy="omit")

PIB și studii: ρ = 0,61, relație puternică, p practic 0. Vârsta și studiile: ρ = −0,23, slabă, dar reală (p < 0,001). PIB și vârsta: ρ = −0,02, p ≈ 0,74, deci nu se deosebește de întâmplare. „Reală” nu înseamnă „puternică”: cu multe observații, chiar și o relație foarte slabă iese semnificativă.

Aceeași logică explică **comparațiile multiple**: dacă se încearcă multe variabile, unele vor trece pragul de 5% doar din întâmplare (una din 20). Ce se găsește explorând se verifică pe alte date înainte de a fi tratat ca rezultat.

**Exercițiu.** Densitatea și șomajul regiunilor: cât de puternică este relația și se deosebește de întâmplare?

In [ ]:
# stats.spearmanr pe coloanele densitate și somaj, cu nan_policy="omit"

## Ce grafic, pentru ce date
| Datele | Graficul | seaborn |
|---|---|---|
| o variabilă numerică | histogramă, densitate, ECDF, boxplot, violin | `histplot`, `kdeplot`, `ecdfplot`, `boxplot`, `violinplot` |
| o variabilă categorică | bare | `countplot` |
| două categorice | tabel de contingență; V Cramér | `heatmap(pd.crosstab())`, `association()` |
| o rețea | noduri și legături | `nx.draw()` |
| numeric × numeric | dispersie; histogramă 2D la multe puncte | `scatterplot`, `histplot(x=, y=)` |
| categoric × numeric | boxplot, violin, puncte pe grupuri | `boxplot(x=, y=)`, `stripplot` |
| o a treia variabilă | culoare, fațete | `hue=`, `col=` |
| valori pe regiuni | hartă coropletă | `harta.plot(column=)` |
| grupuri de regiuni | hartă pe categorii | `harta.plot(column=, categorical=True)` |
| legăturile unei regiuni | hartă cu linii | `ax.plot([x1, x2], [y1, y2])` |

## Deciziile pentru model
| Ce s-a văzut | Decizia pentru C7 |
|---|---|
| SCI: asimetrie pozitivă | scara logaritmică și mediana |
| Svalbard: numitor foarte mic | se exclude |
| lipsurile pe țări | pe câte perechi, spus la fiecare rezultat |
| distanța și granița, legate | două variabile separate |
| migrația contează ca rată | rata la mie, nu numărul |
| partenerii și comunitățile urmează migrația | migrația în model, lângă distanță și graniță |